# Segment neighbors: a sanity check

Do similar-sounding parts of songs come back as nearest neighbors? Pick a moment in a song, listen to
it, then listen to the segments the embeddings say are most similar (section 1). Section 2 measures the
same question over every segment at several segment lengths, so the length can be chosen on evidence.
To change it, edit `WINDOW_S` in `src/synopsist/config.py` and rerun `python scripts/build_catalog.py`;
re-cutting takes seconds because the audio analysis is cached.

**Result (2026-10-10, 35 songs):** passed. Versions of a song find each other 96% of the time, repeats
inside a song are found 85% of the time, and best matches between different songs share tempo, loudness,
and rhythmic density far more often than random pairs. No segment length from 2 to 8 s was clearly better,
so 4 s was kept. Details under section 2.

Run `python scripts/build_catalog.py` first so the database exists.

In [ ]:
import re
import sys
from functools import lru_cache
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "synopsist").is_dir())
sys.path.insert(0, str(root / "src"))

import numpy as np
from IPython.display import Audio, display

from synopsist import db, embed
from synopsist.config import STREAM_DIR, WINDOW_S
from synopsist.index import SegmentIndex
from synopsist.segment import locate, segment_bounds

es = embed.es  # essentia.standard, imported quietly by synopsist.embed
conn = db.connect()
index = SegmentIndex(conn)
tracks = {r["track_id"]: dict(r) for r in conn.execute("SELECT * FROM tracks WHERE analyzed_at IS NOT NULL")}
print(f"{len(tracks)} songs, {len(index)} segments of {WINDOW_S:g} s")

In [ ]:
def clock(seconds):
    return f"{int(seconds // 60)}:{seconds % 60:04.1f}"


@lru_cache(maxsize=8)
def load(track_id, sr=22050):
    return es.MonoLoader(filename=str(STREAM_DIR / tracks[track_id]["file"]), sampleRate=sr)()


def play(track_id, start_s, end_s, pad_s=1.0, sr=22050):
    """A player for part of a song, with pad_s of context on each side."""
    audio = load(track_id, sr)
    return Audio(audio[int(max(0.0, start_s - pad_s) * sr):int((end_s + pad_s) * sr)], rate=sr)


def show(hit):
    print(f"{hit['score']:+.2f}  {hit['title']}  {clock(hit['start_s'])}-{clock(hit['end_s'])}")
    display(play(hit["track_id"], hit["start_s"], hit["end_s"]))

## 1. Pick a moment and listen

Set `TRACK` to any `track_id` from `data/catalog.csv` and `AT_S` to a time in that song. Scores are
cosine similarities after removing what all segments have in common: around 0 means unrelated,
higher means more alike.

In [ ]:
TRACK = "space-rain"
AT_S = 60

bounds = segment_bounds(tracks[TRACK]["duration_s"], WINDOW_S)
idx = int(locate(AT_S, bounds))
print(f"Query: {tracks[TRACK]['title']}, segment {idx} ({clock(bounds[idx][0])}-{clock(bounds[idx][1])})")
display(play(TRACK, *bounds[idx]))

In [ ]:
print("Most similar parts of OTHER songs:")
for hit in index.neighbors(TRACK, idx, k=5):
    show(hit)

In [ ]:
print("Where this part comes back in the SAME song:")
for hit in index.neighbors(TRACK, idx, k=3, other_tracks=False):
    show(hit)

## 2. Do similar-sounding parts come back? (measured)

Three checks over every segment, repeated for several segment lengths. Each length is re-cut from the
analysis cache, so this takes seconds and doesn't touch the database.

- **version**: some songs exist in several versions, recognized here by title (Planetary Fields,
  Tropical Space (Day)). For their segments, is the best match in another song a version of the same
  song? Chance is about 10%.
- **repeat**: songs repeat their sections. Ignoring a segment's direct neighbors, is its best match
  elsewhere in the same song?
- **sound-alike**: for every segment, take its best match in a *different* song and check whether the two
  share tempo (within 4%, or double/half time), key (same, relative, or a fifth apart), loudness, and
  rhythmic density (onsets per second), compared with random pairs of segments. Songs that literally
  reuse material are left out, so this isn't just versions finding each other.

What the first run (35 songs) showed:

| window | version | repeat | same tempo (best / random) | related key | loudness gap (LU) | density gap (octaves) |
|---|---|---|---|---|---|---|
| 2 s | 96% | 86% | 28% / 9% | 50% / 41% | 4.4 / 8.5 | 0.77 / 1.15 |
| 4 s | 96% | 85% | 29% / 9% | 53% / 43% | 4.4 / 8.3 | 0.67 / 1.09 |
| 8 s | 97% | 80% | 29% / 9% | 58% / 45% | 4.2 / 7.8 | 0.62 / 1.04 |

- Similar-sounding parts do come back: a best match is 3 times as likely as a random segment to share
  the tempo, and half as far away in loudness.
- The embeddings capture sound, energy, and groove much more than harmony: key agreement is only about
  10 points above random. Key is stored separately for every segment, so later phases can add it (for
  example for playlist flow) instead of expecting the embeddings to provide it.
- Longer windows match key slightly better but find repeats less often and give the replay heatmap
  coarser detail. No length was clearly better, so 4 s stays.

In [ ]:
analyses = {tid: embed.load_cached(t["file_sha1"]) for tid, t in tracks.items()}
song_ids = sorted(analyses)
# Versions share a title up to " (" or " - ", like "Planetary Fields (Milestone)"
title_song = [re.split(r" \(| - ", tracks[tid]["title"])[0].lower() for tid in song_ids]
has_versions = np.array([title_song.count(s) > 1 for s in title_song])
NOTES = ["C", "C#", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B"]


def fifths(key, scale):
    """Position on the circle of fifths; a minor key sits with its relative major."""
    return ((NOTES.index(key) + (3 if scale == "minor" else 0)) * 7) % 12


def evaluate(window_s, rng=np.random.default_rng(0)):
    # Re-cut every song at this window, from the analysis cache
    song, idx, feats = [], [], []
    for n, tid in enumerate(song_ids):
        analysis = analyses[tid]
        for f in embed.segment_features(analysis, segment_bounds(analysis["duration_s"], window_s)):
            song.append(n)
            idx.append(f["idx"])
            feats.append(f)
    song, idx = np.array(song), np.array(idx)
    vectors = np.array([f["embedding"] for f in feats], dtype=np.float64)
    vectors -= vectors.mean(axis=0)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
    sim = vectors @ vectors.T
    same = song[:, None] == song[None, :]
    elsewhere = np.where(same, -np.inf, sim)

    # 1. Versions: is the best match in another song a version of the same song?
    versioned = np.flatnonzero(has_versions[song])
    best_elsewhere = elsewhere.argmax(axis=1)
    version_found = np.mean([title_song[song[i]] == title_song[song[best_elsewhere[i]]] for i in versioned])

    # 2. Repeats: ignoring itself and its direct neighbors, is a segment's best match in its own song?
    adjacent = same & (np.abs(idx[:, None] - idx[None, :]) <= 1)
    repeat_found = same[np.arange(len(song)), np.where(adjacent, -np.inf, sim).argmax(axis=1)].mean()

    # 3. Sound-alike: compare each segment's best match in a different song with random pairs. Songs that
    #    literally reuse material (half their segments have a near-identical twin, similarity >= 0.8)
    #    are left out, so this isn't just versions finding each other.
    n_songs = len(song_ids)
    has_twin = np.column_stack([(elsewhere[:, song == b] >= 0.8).any(axis=1) for b in range(n_songs)])
    share = np.array([has_twin[song == a].mean(axis=0) for a in range(n_songs)])
    reuse = (share >= 0.5) | (share >= 0.5).T
    blocked = same | reuse[song[:, None], song[None, :]]
    best = np.where(blocked, -np.inf, sim).argmax(axis=1)
    rand_i = np.repeat(np.arange(len(song)), 5)
    rand_j = np.concatenate([rng.choice(np.flatnonzero(~blocked[i]), 5) for i in range(len(song))])

    bpm = np.array([f["bpm"] or np.nan for f in feats])
    fifth = np.array([fifths(f["key"], f["scale"]) if f["key"] else -1 for f in feats])
    loudness = np.array([f["loudness_lufs"] for f in feats])
    density = np.log2(np.maximum([f["onset_rate"] for f in feats], 0.25))

    def agree(i, j):
        ratio = np.abs(np.log2(bpm[i] / bpm[j]))
        gap = np.abs(fifth[i] - fifth[j]) % 12
        known = (fifth[i] >= 0) & (fifth[j] >= 0)
        return np.array([
            np.mean(np.minimum(ratio, np.abs(ratio - 1)) < np.log2(1.04)),  # same tempo within 4%, or double/half
            np.mean(np.minimum(gap, 12 - gap)[known] <= 1),  # same key, relative key, or a fifth apart
            np.mean(np.abs(loudness[i] - loudness[j])),  # loudness gap, LU
            np.mean(np.abs(density[i] - density[j])),  # onset density gap, octaves
        ])

    reused = [" / ".join(sorted(tracks[song_ids[t]]["title"] for t in pair)) for pair in zip(*np.nonzero(np.triu(reuse)))]
    return len(song), version_found, repeat_found, agree(np.arange(len(song)), best), agree(rand_i, rand_j), reused


print("                            -------- best match in another song vs. a random pair --------")
print("window segments version repeat  same tempo   related key  loudness gap  density gap")
for window_s in [2, 3, 4, 6, 8]:
    n, version_found, repeat_found, best, rand, reused = evaluate(window_s)
    print(f"{window_s:4g} s {n:8d} {version_found:7.0%} {repeat_found:6.0%}  {best[0]:4.0%} / {rand[0]:<4.0%} "
          f"{best[1]:4.0%} / {rand[1]:<4.0%} {best[2]:4.1f} / {rand[2]:<4.1f}  {best[3]:4.2f} / {rand[3]:.2f}")
    if window_s == WINDOW_S:
        current_reuse = reused
print(f"\nLeft out as reused material at {WINDOW_S:g} s: " + "; ".join(current_reuse))

## 3. Which songs share material?

For each song, the share of its segments whose best match in another song lands in a given other
song. Versions should top the list. Strong links between different songs matter for recommendations:
recommending another version of a song someone already replays isn't a real recommendation.

In [ ]:
sim = index.vectors @ index.vectors.T
same_song = index.track_ids[:, None] == index.track_ids[None, :]
best_song = index.track_ids[np.where(same_song, -np.inf, sim).argmax(axis=1)]

links = {}
for tid, other in zip(index.track_ids, best_song):
    links[tid, other] = links.get((tid, other), 0) + 1 / np.count_nonzero(index.track_ids == tid)
for (a, b), share in sorted(links.items(), key=lambda item: -item[1])[:12]:
    print(f"{share:4.0%} of {tracks[a]['title']}  ->  {tracks[b]['title']}")